# Creation of the agent
## Uses
An agent is a chain that gets the user input, proceses it an LLM to decide which tool to call and what inputs to pass to it and returns an answer. 
Depending on the query it receives, the agent needs to decide between: 
- returns a summary of the weather in madrid 
- basic question for the LLM 
- propose a more complete prompt based on what is suggested

For our agent we mainly need: 
- the agent model (i.e:openais gpt-3.5-turbo-1106) is the LLM that will act as the agent’s brain, deciding which tools to call and what inputs to pass them from an environment variable
- its prompt template which define that is an agent and how to behave with the prompt. We'll get from LangChain Hub: https://smith.langchain.com/hub/hwchase17/openai-functions-agent
- the tools which the agent will interact with the world. These can be built-in LangChain tools (search, calculator, file operations), api integrations, ...
- the agent executor: this orchestrates the interaction between the LLM, tools, and prompt template.

### The agent prompt
Instead of defining your own prompt for the agent, which you can certainly do, we will load a predefined prompt from LangChain Hub. LangChain hub lets you upload, browse, pull, test, and manage prompts. In this case, the default prompt for OpenAI agents, hwchase17/openai-functions-agent, works great. You can use it, as-is, but if you want top know how it works continue reading

A simpler prompt could be: 
```python
from langchain.prompts import ChatPromptTemplate

# Define prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools."),
    ("user", "{input}"),
    ("assistant", "{agent_scratchpad}")
])
```
But the one hwchase17/openai-functions-agent is a little more sophisticated. 

```python
ChatPromptTemplate(
    input_variables=['agent_scratchpad', 'input'], 
    input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, 
                                            langchain_core.messages.human.HumanMessage, 
                                            langchain_core.messages.chat.ChatMessage, 
                                            langchain_core.messages.system.SystemMessage, 
                                            langchain_core.messages.function.FunctionMessage, 
                                            langchain_core.messages.tool.ToolMessage]], 
                'agent_scratchpad': typing.List[... again all messages types ...]
    }, 
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                    input_variables=[], 
                    template='You are a helpful assistant'
                )
        ), 
        MessagesPlaceholder(
            variable_name='chat_history', 
            optional=True
        ), 
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                    input_variables=['input'], 
                    template='{input}'
                )
        ), 
        MessagesPlaceholder(
            variable_name='agent_scratchpad'
        )
    ]

```
**Overall Structure**

This template creates a structured conversation flow with four main components arranged in order:

**System instructions**
- System message: "you are a helpful assistant"
- Chat history (optional)
- Current user input
- Agent scratchpad

**Key Components Explained**

1. agent_scratchpad
This is the "working memory" of the agent during tool usage. It contains the sequence of:
- Tool calls the agent makes
- Tool responses from those calls
- Agent reasoning between tool uses

Example flow:
```shell
User: "What's the weather in Paris?"
Agent: I'll check the weather for you.
[Tool Call: weather_api("Paris")]
[Tool Response: "22°C, sunny"]
Agent: The weather in Paris is 22°C and sunny.
```
The scratchpad captures this entire reasoning chain, allowing the agent to:

Track what tools it has used
- Remember tool results
- Make multi-step reasoning decisions
- Avoid repeating the same tool calls

2. chat_history
This maintains the conversation context across multiple turns. It stores the complete conversation history as a list of messages (Human, AI, System, etc.).
Purpose:
- Enables follow-up questions ("What about tomorrow?")
- Maintains context ("Paris" from previous question)
- Allows referencing earlier parts of conversation

3. Input Variables & Types
- input: The current user question/request
- agent_scratchpad: The tool usage history for current turn

4. Message Flow

- SystemMessagePromptTemplate: Sets the agent's role ("You are a helpful assistant")
- MessagesPlaceholder(chat_history): Injects previous conversation turns (marked optional=True)
- HumanMessagePromptTemplate: The current user input
- MessagesPlaceholder(agent_scratchpad): The agent's tool usage and reasoning for this turn

**Why This Structure?**

This template enables ReAct-style reasoning (Reasoning + Acting):

- Agent receives input
- Can use tools to gather information
- Reasons about tool results
- Provides final response
- All while maintaining conversation context

The scratchpad is essential because it allows the LLM to see its own "thought process" and tool interactions, enabling more sophisticated multi-step problem solving.

In [22]:
import dotenv
import os
from langchain import hub
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
AGENT_MODEL = os.getenv("OPENAI_AGENT_MODEL")
chat_model = ChatOpenAI(model=AGENT_MODEL,temperature=0,
)
agent_prompt = hub.pull("hwchase17/openai-functions-agent")
agent_prompt

/data/AssistantStudies/.venv/lib/python3.11/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

### The Tools

Your agent has initially three tools available to it: 
- returns a summary of the weather in madrid: thats a call to a function
- basic question for the LLM 
- propose a more complete prompt based on what is suggested
The basic question and propose prompt tools call .invoke() from their respective chains, while weather call the functions you defined. 
Notice that many of the tool descriptions have few-shot prompts, telling the agent when it should use the tool and providing it with an example of what inputs to pass.

As with chains, good prompt engineering is crucial for your agent’s success. You have to clearly describe each tool and how to use it so that your agent isn’t confused by a query and choose the wrong tool.

### get_basic_question_chain tool

In [17]:
from langchain.prompts import (
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate,
)

def get_basic_question_chain():
    CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL") # gpt-4o
    chat_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    basic_question_system_template_str = """You are a helpful assistant. 
    Your job is to answer to the user his questions to the best of your hability. 
    before your answer, asses the uncertainty of your answer. If its greater then 0.3, ask him to redo the question, 
    indicating what clarifications he need made so you can answer better.
    """

    system_prompt = SystemMessagePromptTemplate(
        prompt=PromptTemplate.from_template(basic_question_system_template_str)
    )

    human_prompt = HumanMessagePromptTemplate(
        prompt=PromptTemplate.from_template("{question}")
    )

    messages = [system_prompt, human_prompt]
    template = ChatPromptTemplate.from_messages(messages)

    basic_question_chain = template | chat_model

    return basic_question_chain

In [27]:
from weather.openmeteo import get_tempt_prompt
def get_current_weather_chain():
    CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL") # gpt-4o
    chat_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    system_template_str = f"You are a meteorologist. Your job is to answer to what will be todays weather. You know that today's weather forecast is {get_tempt_prompt()}."

    system_prompt = SystemMessagePromptTemplate(
        prompt=PromptTemplate.from_template(system_template_str)
    )

    human_prompt = HumanMessagePromptTemplate(
        prompt=PromptTemplate.from_template("{question}")
    )

    messages = [system_prompt, human_prompt]
    template = ChatPromptTemplate.from_messages(messages)

    question_chain = template | chat_model

    return question_chain

In [28]:
from langchain.agents import (
    create_openai_functions_agent,
    Tool,
    AgentExecutor,
)

basic_question_chain = get_basic_question_chain()
current_weather_chain = get_current_weather_chain()

tools = [
    Tool(
        name="BasicQuestions",
        func=basic_question_chain.invoke,
        description="""Useful when you need to answer all kind of questions except if they are about the
        weather today, or about generate propmpts based on the instructions in the received one. Use the
        entire prompt as input to the tool. For instance, if the prompt is
        "How big is the city of Oviedo?", the input should be
        "How big is the city of Oviedo?".
        """,
    ),
    Tool(
        name="WeatherSummary",
        func=current_weather_chain.invoke,
        description="Use when asked about current weather today.",
    ),
]

In [23]:

sebastian_agent = create_openai_functions_agent(
    llm=chat_model,
    prompt=agent_prompt,
    tools=tools,
)

sebastian_agent_executor = AgentExecutor(
    agent=sebastian_agent,
    tools=tools,
    return_intermediate_steps=True,
    verbose=True,
)



In [43]:
response = sebastian_agent_executor.invoke(
     {"input": "tell me the latitude and longitude of New York in between brackets, for example gijon is [43.5357, -5.6615]?"}
 )

print(f'{response["input"]=}')
print(f'{response["output"]=}')
response["intermediate_steps"]



> Entering new AgentExecutor chain...
The latitude and longitude of New York City are [40.7128, -74.0060].

> Finished chain.
response["input"]='tell me the latitude and longitude of New York in between brackets, for example gijon is [43.5357, -5.6615]?'
response["output"]='The latitude and longitude of New York City are [40.7128, -74.0060].'


[]